# DRIVE


In [ ]:
#Your google drive is made accesible to Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive/')
    #%cd /content/drive/MyDrive/tfg_guillermo/code/Transporte lineal 2 (sin model.fit).ipynb
    %cd /content/drive/MyDrive/TFGs_TFM_2023-2024/TFGs_propuestas_Física/Guillermo_Gracia_Rebullida/tfg_guillermo/code
    %ls -lht
    # To import own packages set local path in packages syspath
    import sys
    sys.path.insert(0,"./")
except ImportError:
    print("You are not in google.colab!!")
    import os
    # List files in the current working directory
    files_in_directory = os.listdir()
    for file in files_in_directory:
        print(file)

Mounted at /content/drive/
[Errno 2] No such file or directory: '/content/drive/MyDrive/TFGs_TFM_2023-2024/TFGs_propuestas_Física/Guillermo_Gracia_Rebullida/tfg_guillermo/code'
/content
total 8.0K
drwx------ 6 root root 4.0K Apr 10 15:54 drive/
drwxr-xr-x 1 root root 4.0K Apr  8 13:27 sample_data/


# Librerias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import sympy as sp
from sympy.utilities.lambdify import lambdify
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense
import imageio.v2 as imageio

# Condiciones iniciales

In [ ]:
#condiciones iniciales
v=0.07
v_tensor=tf.constant([[0.07]],dtype=tf.float32)

x, v = sp.symbols('x v ')
phi0 = sp.exp(-x**2/(4*v))+sp.exp(-(x-2*np.pi)/(4*v))
phi0_prima = phi0.diff(x)
u0 = -2 * v * (phi0_prima / phi0) + 4.0

u0 = lambdify((x), u)


def phi0(x):
  return np.exp(-x**2/(4*v))+np.exp(-(x-2*np.pi)/(4*v))

def u0(x):
  return -2*v/phi0(x)*np.diff(phi0,x)+4.0

# Solución

In [ ]:
#solucion

def phi(x,t):
  return sp.exp(-(x-4*t)**2/(4*v*(t+1)))+sp.exp(-(x-4*t-2*np.pi)**2/(4*v*(t+1)))

x, nu = sp.symbols('x nu ')
phi0 = (sp.exp(-(x - 4 * t)**2 / (4 * nu * (t + 1))) +
       sp.exp(-(x - 4 * t - 2 * sp.pi)**2 / (4 * nu * (t + 1))))

def u(x,t):
  return -2*v/phi(x,t)*sp.diff(phi,x)+4.0


# Datos

In [ ]:
#dominio

xmin=0
xmax=2*sp.pi
tmin=0
tmax=1
n_train=100
n_train0=125
n_border=50
N_STEPS=10000
log_each=500

# Entrenamiento PINN

In [ ]:
#parametros redes
nInput=2
nOutput=1
nhiddenlayers=4
nnhiddenlayers=100

#hiperparametros
lr=0.001

#nn

model=Sequential()
model.add(Input(shape=(nInput,)))
for i in range(0,nhiddenlayers):
  model.add(Dense(nnhiddenlayers,activation='sigmoid'))
model.add(Dense(1,activation=None))

# Metrica, perdida y optimizador
loss = tf.keras.losses.MeanSquaredError()
metrics = tf.keras.metrics.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(lr)

model.compile(loss=loss,optimizer=optimizer,metrics=[metrics])

ceros = tf.zeros(n_border,dtype=tf.float32)
ones = tf.ones(n_border,dtype=tf.float32)
two_pis= 2*np.pi*ones

x_train=np.random.uniform(xmin,xmax,n_train)
t_train=np.random.uniform(tmin,tmax,n_train)
tensor_train=np.column_stack((x_train,t_train))
tensor_train=tf.convert_to_tensor(tensor_train,dtype=tf.float32)

x_train0=np.random.uniform(xmin,xmax,size=n_train0)
t_train0=np.zeros(n_train0)
tensor_train0=np.column_stack((x_train0,t_train0))
tensor_train0=tf.convert_to_tensor(tensor_train0,dtype=tf.float32)

t = tf.convert_to_tensor(np.random.uniform(tmin,tmax,n_border),dtype=tf.float32)
tensor_border1 = tf.stack([ceros,t], axis=-1)
tensor_border2 = tf.stack([two_pis,t], axis=-1)

u00 = tf.transpose(u0(x_train0))


#@tf.function
def model_fit(N_STEPS):
    history = {'loss': [0.0], 'PL': [0.0], 'ZL': [0.0], 'BL': [0.0]}
    for step in range(0,N_STEPS+1):
        with tf.GradientTape(persistent=True) as tape:
            with tf.GradientTape(persistent=True) as tape2:
                tape2.watch(tensor_train)
                with tf.GradientTape(persistent=True) as tape3:
                    tape3.watch(tensor_train)
                    tape3.watch(tensor_train0)
                    tape3.watch(tensor_border1)
                    tape3.watch(tensor_border2)

                    uNN = model(tensor_train,training=True)
                    uNN0 = model(tensor_train0,training=True)
                    uNNborder1 = model(tensor_border1,training=True)
                    uNNborder2 = model(tensor_border2,training=True)

                partial_derivatives1 = tape3.gradient(uNN,tensor_train)
                partial_derivatives_x = partial_derivatives1[:, 0:1]  # Extraer componente

            partial_derivatives2 = tape2.gradient(partial_derivatives_x, tensor_train)
            partial_loss = model.compiled_loss(partial_derivatives1[:,1]+uNN*partial_derivatives1[:,0],v_tensor*partial_derivatives2[:,0])
            zero_loss = model.compiled_loss(uNN0, u00)
            border_loss = model.compiled_loss(uNNborder1,uNNborder2)
            loss=tf.cast(partial_loss, dtype=tf.float32)+tf.cast(zero_loss, dtype=tf.float32)+tf.cast(border_loss, dtype=tf.float32)

        gradients=tape.gradient(loss,model.trainable_weights)
        model.optimizer.apply_gradients(zip(gradients, model.trainable_weights))

        if step % log_each == 0:
            metrics= {m.name: m.result() for m in model.metrics}
            metrics['PL']= partial_loss.numpy()
            metrics['ZL']=zero_loss.numpy()
            metrics['BL']=border_loss.numpy()
            print("step=",step,metrics)
            for key in history.keys():
                history[key].append(metrics[key])
    return history
    '''
    if step % log_each == 0:
      print(f'{step}/{N_STEPS} partial_loss {partial_loss:.5f} zero_loss {zero_loss:.5f}  border_loss {border_loss:.5f}')
    '''

history=model_fit(N_STEPS)

/usr/local/lib/python3.10/dist-packages/sympy/core/function.py:1233: SymPyDeprecationWarning: 

The string fallback in sympify() is deprecated.

To explicitly convert the string form of an object, use
sympify(str(obj)). To add define sympify behavior on custom
objects, use sympy.core.sympify.converter or define obj._sympy_
(see the sympify() docstring).

sympify() performed the string fallback resulting in the following string:

'<function phi0 at 0x7dab1c770e50>'

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-sympify-string-fallback
for details.

This has been deprecated since SymPy version 1.6. It
will be removed in a future version of SymPy.

  expr = sympify(expr)


SympifyError: Sympify of expression 'could not parse '<function phi0 at 0x7dab1c770e50>'' failed, because of exception being raised:
SyntaxError: invalid syntax (<string>, line 1)

In [ ]:
ones = tf.ones(n_border,dtype=tf.float32)
two_pis= 2*np.pi*ones

print(ones)
print(two_pis)

tf.Tensor(
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1.], shape=(50,), dtype=float32)
tf.Tensor(
[6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855 6.2831855
 6.2831855], shape=(50,), dtype=float32)
